In [2]:
import os
import pickle
from PIL import Image
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1

In [3]:
def load_and_transform_image(image_path):
    transform_list = [
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ]
    
    transform = transforms.Compose(transform_list)
    
    try:
        image = Image.open(image_path).convert('RGB')
        return transform(image)
    except Exception as e:
        print(f"Error memuat gambar {image_path}: {e}")
        return torch.zeros(3, 256, 256)
        

In [4]:
class FacesDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (str): Direktori root yang berisi subfolder untuk setiap label.
            transform (callable, optional): Transformasi yang akan diterapkan pada gambar.
            augment (bool): Jika True, terapkan augmentasi pada gambar.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # Enumerasi semua subfolder dan gambar di dalamnya
        for label in os.listdir(root_dir):
            label_dir = os.path.join(root_dir, label)
            if not os.path.isdir(label_dir):
                continue
            for img_name in os.listdir(label_dir):
                if img_name.lower().endswith('.ppm'):
                    full_path = os.path.join(label_dir, img_name)
                    if os.path.isfile(full_path):
                        self.image_paths.append(full_path)
                        self.labels.append(label)
                    else:
                        print(f"File tidak ditemukan: {full_path}")
        
        if len(self.image_paths) == 0:
            raise RuntimeError(f"Tidak ada file .ppm ditemukan di {root_dir}")
                    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        if self.transform:
            image = self.transform(img_path)
        else:
            image = load_and_transform_image(img_path)
        return image, label

In [5]:
def generate_embeddings(train_dir, batch_size=32, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    Menghasilkan embedding untuk semua gambar dalam direktori train.
    
    Args:
        train_dir (str): Path ke direktori train.
        batch_size (int, optional): Ukuran batch untuk DataLoader.
        device (str, optional): Device untuk menjalankan model ('cuda' atau 'cpu').
        
    Returns:
        dict: Dictionary dengan label sebagai key dan list embedding sebagai value.
    """
    try:
        dataset = FacesDataset(root_dir=train_dir)
    except RuntimeError as e:
        print(e)
        return {}
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    # Inisialisasi model FaceNet pra-terlatih
    model = InceptionResnetV1(pretrained='vggface2').eval().to(device)
    
    embeddings_dict = {}
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Membuat Embedding"):
            images = images.to(device)
            embeddings = model(images)
            embeddings = embeddings.cpu().numpy()
            
            for label, embedding in zip(labels, embeddings):
                if label not in embeddings_dict:
                    embeddings_dict[label] = []
                embeddings_dict[label].append(embedding)
    
    return embeddings_dict

In [ ]:
def main():
    train_directory = 'train'  # Ganti dengan path ke direktori train Anda
    
    if not os.path.exists(train_directory):
        print(f"Direktori train tidak ditemukan: {train_directory}")
        return
    
    embeddings = generate_embeddings(train_directory)
    
    if embeddings:
        # Simpan embeddings ke file pickle
        with open('embeddings_no_augmentation.pkl', 'wb') as f:
            pickle.dump(embeddings, f)
        
        print("Embedding berhasil disimpan dalam embeddings_no_augmentation.pkl")
    else:
        print("Tidak ada embedding yang disimpan karena terjadi error.")

In [7]:
if __name__ == "__main__":
    main()

Membuat Embedding: 100%|██████████| 13/13 [00:38<00:00,  3.00s/it]

Embedding berhasil disimpan dalam embeddings.pkl
